In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [16]:
from langchain_openai import ChatOpenAI
from openai import OpenAI

In [6]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph,START,END,MessagesState

In [ ]:
from langgraph.prebuilt import ToolNode,tools_condition
from IPython.display import Image, display
from langgraph.checkpoint.memory import MemorySaver
from pinecone.grpc import PineconeGRPC
from qdrant_client import QdrantClient
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [31]:
pinecone_api_key=os.getenv("PINECONE_API_KEY")
pinecone_index_name=os.getenv("PINECONE_INDEX_NAME")
openai_api_key=os.getenv("OPENAI_API_KEY")
pc=PineconeGRPC(api_key=pinecone_api_key)
index_description = pc.describe_index(name=pinecone_index_name)
index = pc.Index(host=index_description.host)
qdrant_url=os.getenv("Qdrant_host")
qdrant_api_key = os.getenv("Qdrant_api_key")
qdrant_client = QdrantClient(
    url=qdrant_url,
    api_key=qdrant_api_key)

In [92]:
def HR_retrieval(question):
    """
    This function used to retrieve similar documents related to HR policy related question.
    It will first convert the question into vector and
    then use the vector to retrieve similar documents from pinecone collection.
    """
    embedding_client = OpenAI(api_key=openai_api_key)
    query_emded=embedding_client.embeddings.create(
        input=question,
        model="text-embedding-3-small",
        encoding_format="float"
    )
    query_vector=query_emded.data[0].embedding
    answer = index.query(
        vector=query_vector,
        top_k=2,
        include_metadata=True
    )
    return "\n\n".join(i.metadata["text"] for i in answer.matches)

In [90]:
def law_retrieval(question):
    """
    This function used to retrieve similar documents related to Court case related question.
     It will first convert the question into vector and
    then use the vector to retrieve similar documents from qdrant collection.
    """

    embedding_client = OpenAI(api_key=openai_api_key)
    query_emded=embedding_client.embeddings.create(
        input=question,
        model="text-embedding-3-small",
        encoding_format="float"
    )
    query_vector=query_emded.data[0].embedding
    answer=qdrant_client.query_points(
        query=query_vector,
        limit=2,
        collection_name="law_policy_collection"
    )
    return "\n\n".join(i.payload["text"] for i in answer.points)

In [42]:
tool_kit = [law_retrieval,HR_retrieval]

In [41]:
llm = llm=ChatOpenAI(model="gpt-4o-mini")

In [43]:
llm_with_tools = llm.bind_tools(tool_kit)

In [ ]:
router_prompt = """
You are a query classification assistant.

Your task is to classify the user’s question into exactly ONE of the following categories 
based strictly on its intent.

Available knowledge sources:
1. HR_POLICY  → Questions related to company HR policies (stored in Pinecone index).
2. LAW_CASE   → Questions related to historical court cases or legal verdicts 
(stored in Qdrant collection).
3. UNKNOWN    → Questions that do not clearly belong to HR policies or legal case verdicts.

Instructions:
- Carefully analyze the user question.
- Classify based on the primary intent of the question.
- Do NOT answer the question.
- Do NOT explain your reasoning.
- Do NOT hallucinate.
- Output ONLY one of the following exact labels:
  HR_POLICY
  LAW_CASE
  UNKNOWN

User Question:
{question}
"""

In [70]:
def router_decision(question):
    """
    This Function classifies the user question into 3 category using LLM
    1. HR Ploicy
    2. Law case
    3.  
    """
    prompt = ChatPromptTemplate.from_template(router_prompt)
    chain = prompt | llm | StrOutputParser()
    response = chain.invoke({"question":question})
    return response

In [72]:
router_decision("what is Miranda case about?")

'LAW_CASE'

In [ ]:
def generation(chunks):